# 明实录 TPU v5e-8 试跑（vLLM + Qwen3-30B-A3B-Instruct-2507）

**目标**：验证 Kaggle TPU 上 vLLM 能不能跑 Qwen3-30B-A3B（MoE），跑通就拉 P7 切片 500 段做速度对比。

**预期失败点**（每一步都可能挂）：
1. ~~vLLM TPU 后端依赖与 Kaggle JAX/torch_xla 不兼容~~ → v2 改走 `tpu-inference` 插件 + `libtpu` PyPI 通道
2. Qwen3-30B-A3B 在 vLLM TPU backend 上 MoE all-to-all 路由不支持
3. bf16 加载 ~60 GB 模型在 TPU v5e-8 总 128 GB HBM 里塞不下（tensor_parallel=8 时每片 ~7.5 GB 模型 + KV cache）
4. HF 仓库限速 / 模型下载超时

**Accelerator**：右栏选 `TPU VM v5e-8`（9 小时上限）

## 1. TPU 健康检查

In [ ]:
import os, sys, subprocess
print('Python:', sys.version)
print('\n--- /kaggle/input/ ---')
!ls /kaggle/input/ 2>&1
print('\n--- TPU 环境 ---')
for k in ['TPU_NAME', 'XLA_FLAGS', 'PJRT_DEVICE', 'TPU_LIBRARY_PATH']:
    print(f'  {k} = {os.environ.get(k, "(unset)")}')

# 看看 torch_xla 当前能识别多少设备
try:
    import torch_xla
    print(f'\ntorch_xla 版本: {torch_xla.__version__}')
    print(f'PJRT device count: {torch_xla.runtime.device_count()}')
except Exception as e:
    print(f'torch_xla 检测失败: {e}')

# JAX 也看下
try:
    import jax
    print(f'\nJAX 版本: {jax.__version__}')
    print(f'JAX 设备: {jax.devices()}')
except Exception as e:
    print(f'JAX 检测失败: {e}')

## 2. 装 vLLM TPU 后端 + 插件

vLLM ≥ 0.20 把 TPU backend 从主仓库剥离，改成独立的 `tpu-inference` 插件包。
另外 `jaxlib[tpu]` 这个 extra 在 jaxlib 0.5+ 之后没了，要走 `libtpu` PyPI 路径。

依赖：`vllm` + `tpu-inference` + `libtpu`（Kaggle TPU runtime 预装了 torch_xla 和 jax，一般不用动）。

In [ ]:
# vLLM ≥ 0.20 把 TPU backend 移到独立插件 tpu-inference
# 老 cell 装 'jaxlib[tpu]' 在 jaxlib 0.5+ 上已无 extra，必挂；改用 libtpu PyPI 通道
%pip install -q vllm
%pip install -q tpu-inference -f https://storage.googleapis.com/libtpu-releases/index.html

import importlib, vllm
print(f'vLLM 版本: {vllm.__version__}')

# tpu_inference 插件健康检查（包名带连字符，导入名带下划线）
# 这里 OK 加上 cell 启动时 tpu_info 打印的 num_chips=8，就等于 TPU 已 ready
try:
    importlib.import_module('tpu_inference')
    print('tpu_inference 插件: OK')
except ImportError as e:
    raise RuntimeError(f'tpu_inference 没装上 — vLLM TPU backend 起不来：{e}')

# 注：vLLM 0.22 把 vllm.platforms.current_platform / get_current_platform 都拆掉了，
# 没有稳定的轻量探针，干脆不再检测；上面 tpu_inference OK + 启动日志的 tpu_type 已足够

## 3. 加载 Qwen3-30B-A3B-Instruct-2507（bf16）

MoE 模型，30B 总参 / 3B 激活。bf16 约 60 GB，分到 8 个 TPU v5e chip（每 chip 16 GB HBM，共 128 GB）。

In [ ]:
import time
from vllm import LLM, SamplingParams

MODEL_ID = 'Qwen/Qwen3-30B-A3B-Instruct-2507'

t0 = time.time()
llm = LLM(
    model=MODEL_ID,
    tensor_parallel_size=8,         # 吃满 8 块 v5e
    max_model_len=4096,
    dtype='bfloat16',
    trust_remote_code=True,
    enforce_eager=False,            # 让 XLA 编译图（warmup 慢、稳态快得多）
    max_num_seqs=512,               # 同时排队请求数
    max_num_batched_tokens=16384,   # 每步喂的 token 数；越大矩阵越饱
    gpu_memory_utilization=0.95,    # 多挤点 HBM 给 KV cache（TPU 上也走这个参数）
    download_dir='/tmp/hf_cache',
)
print(f'模型加载 OK，耗时 {(time.time()-t0)/60:.1f} 分钟')


## 4. 测一条样本，确认输出合理

In [ ]:
FEWSHOT_PROMPT = '''下面是给古文加中文标点的任务。

规则：
1. 保留所有原字，只在字间插入标点（，。：；？！、《》「」）。
2. 段落必须以句末符号结尾（。 ？ ！ 」 』 ）之一）；不要以 ， ： 、 ； 收尾。
3. 不要连续两个标点（除了 。」 ？」 ！」 这类引号闭合）。
4. 「」必须成对出现，《》必须成对出现。

无标点：自古帝王之有天下其言行政治必有史臣纪载以垂鉴戒此古今之盛典朝廷之先务也
有标点：自古帝王之有天下，其言行政治，必有史臣纪载，以垂鉴戒，此古今之盛典，朝廷之先务也。

无标点：奉天门常朝御座后内官持一小扇金黄绢以裹之尝闻一老将军云非扇也其名卓影辟邪永乐间外国所进
有标点：奉天门常朝，御座后内官持一小扇，金黄绢以裹之。尝闻一老将军云：「非扇也，其名卓影辟邪，永乐间外国所进。」

无标点：{raw}
有标点：'''

test_raw = '朝廷每端午节赐朝官吃糕糭于午门外酒数行而出文职大臣仍从驾幸后苑观武臣射栁事毕'

sampling = SamplingParams(
    temperature=0,
    max_tokens=512,
    stop=['\n\n无标点：', '\n无标点：', '<|im_end|>', '<|endoftext|>'],
)

t0 = time.time()
result = llm.generate([FEWSHOT_PROMPT.format(raw=test_raw)], sampling)
elapsed = time.time() - t0
out = result[0].outputs[0].text.strip()
print(f'耗时 {elapsed:.1f}s')
print(f'输出: {out!r}')

## 5. 跑 P7 前 50 段做速度 benchmark

和 T4x2 的 0.47 段/秒对比。

In [ ]:
# 拉数据：P7 切片
%pip install -q kagglehub
import kagglehub, json
ds_path = kagglehub.dataset_download('canhuiliphy/mingshilu-prechunked-p78')
rows = [json.loads(l) for l in open(f'{ds_path}/mingshilu-p7-prechunked.jsonl', 'r', encoding='utf-8') if l.strip()]
print(f'P7 总 {len(rows)} 行')

# 先 warmup 50 条让 XLA 把 graph 编译完
N_WARMUP = 50
prompts_warmup = [FEWSHOT_PROMPT.format(raw=r['raw']) for r in rows[:N_WARMUP]]
print(f'warmup {N_WARMUP} 条（首次编译会慢）...')
t0 = time.time()
_ = llm.generate(prompts_warmup, sampling)
print(f'warmup 耗时 {time.time()-t0:.1f}s')

# 正式 benchmark：500 条一次塞进去
N_BENCH = 500
samples = rows[N_WARMUP : N_WARMUP + N_BENCH]
prompts = [FEWSHOT_PROMPT.format(raw=r['raw']) for r in samples]
print(f'\nbenchmark {N_BENCH} 条，平均 prompt 长度 {sum(len(p) for p in prompts)/len(prompts):.0f} 字符')

t0 = time.time()
results = llm.generate(prompts, sampling)
elapsed = time.time() - t0
rate = N_BENCH / elapsed
print(f'\n{N_BENCH} 段批处理耗时 {elapsed:.1f}s，速率 {rate:.2f} 段/秒')
print(f'对比 T4x2 基线 0.47 段/秒，TPU 加速比 {rate/0.47:.1f}×')
print(f'21,605 行 P7 推算耗时: {21605/rate/60:.0f} 分钟')

# 抽 4 条看输出对不对
for i in [0, 100, 250, 499]:
    if i >= len(results): continue
    out = results[i].outputs[0].text.strip()
    raw = samples[i]['raw']
    print(f'\n[{i}] raw ({len(raw)}字): {raw[:60]}...')
    print(f'    out ({len(out)}字): {out[:80]}...')


## 结论

看 4 节的速率 vs T4x2 基线 0.47 段/秒：
- 如果 > 2×：TPU 路径有前途，下一步写完整跑批 notebook（带 rescue cascade + checkpoint）
- 如果 < 2× 或挂掉：放弃 TPU，继续 T4